# Optimization Input

How to access aggregation results for energy system optimization models.

This notebook shows how to access:
- Cluster weights (occurrence counts)
- Cluster assignments (period ordering)
- Typical period data

Import pandas and the relevant time series aggregation class

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

import tsam
from tsam import ClusterConfig, ExtremeConfig

### Input data 

Read in time series from testdata.csv with pandas

In [2]:
raw = pd.read_csv("testdata.csv", index_col=0)

Transform the index to a datetime index

In [3]:
raw.index = pd.to_datetime(raw.index)

Plot raw data

In [4]:
fig = px.line(
    raw, facet_row="variable", labels={"index": "Time", "value": ""}
).update_yaxes(matches=None)
fig.show()

### Aggregate the data

Aggregate to typical weeks, including days with minimum temperature and maximum load as extreme periods.

In [5]:
result = tsam.aggregate(
    raw,
    n_clusters=5,
    period_duration=24 * 7,  # Weekly periods
    cluster=ClusterConfig(method="hierarchical"),
    extremes=ExtremeConfig(
        method="new_cluster",
        min_value=["T"],
        max_value=["Load"],
    ),
)

Create the typical periods

In [6]:
cluster_representatives = result.cluster_representatives
cluster_representatives.head()

GHI         T      Wind        Load
  timestep                                     
0 0         0.0  0.220495  0.307872  416.965693
  1         0.0  0.008529  0.000000  409.436001
  2         0.0 -0.203436  0.000000  403.636735
  3         0.0 -0.097453  0.205248  404.487067
  4         0.0 -0.203436  0.307872  410.103512

### Show the resulting order  of aggregated periods

Calculates how the original index is represented by the cluster index

In [7]:
index_matching = result.assignments
index_matching.head()

,period_idx,timestep_idx,cluster_idx
2009-12-31 23:30:00,0,0,3
2010-01-01 00:30:00,0,1,3
2010-01-01 01:30:00,0,2,3
2010-01-01 02:30:00,0,3,3
2010-01-01 03:30:00,0,4,3


Plot the appearance of the 5+2 aggregated periods in the original timeframe

In [8]:
visualization_df = pd.DataFrame(
    0, index=index_matching.index, columns=result.period_index
)
for col in visualization_df.columns:
    visualization_df.loc[index_matching["cluster_idx"] == col, col] = 1

In [9]:
fig = px.area(
    visualization_df,
    labels={"index": "Time", "value": "Occurrence"},
    color_discrete_sequence=px.colors.sample_colorscale(
        "Viridis", len(visualization_df.columns)
    ),
)
fig.show()

### Get input for potential energy system optimization

**i. cluster_counts** - The occurrence count of each typical period for weighting in the objective function.

Note: Period three is only partially evaluated since its appearance at the end of the year exceeds the original time series.

In [10]:
result.cluster_counts

{0: 10.0, 1: 17.0, 2: 7.0, 3: 4.142857142857142, 4: 8.0, 5: 3.0, 6: 3.0}

In [11]:
weights = pd.Series(result.cluster_counts)
fig = px.bar(
    x=weights.index,
    y=weights.values,
    labels={"x": "Period index", "y": "Number of occurence"},
)
fig.show()

**ii. Accessing period data by index**
<br>Access aggregated time series values using period and time step indices from the cluster_representatives DataFrame.

In [12]:
# Access a specific value: GHI for cluster period 3, timestep 12
result.cluster_representatives.loc[(3, 12), "GHI"]

np.float64(132.86908868164954)

Alternatively this is given as data frame

In [13]:
result.cluster_representatives.head()

GHI         T      Wind        Load
  timestep                                     
0 0         0.0  0.220495  0.307872  416.965693
  1         0.0  0.008529  0.000000  409.436001
  2         0.0 -0.203436  0.000000  403.636735
  3         0.0 -0.097453  0.205248  404.487067
  4         0.0 -0.203436  0.307872  410.103512

**iii. cluster_assignments**
<br> The order of the typical periods to represent the original time series, e.g., to model seasonal storage.

In [14]:
result.cluster_assignments

array([3, 2, 5, 6, 6, 2, 5, 5, 3, 0, 0, 0, 4, 0, 0, 4, 1, 4, 4, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 1, 4, 4, 0, 0, 3, 0, 0,
       0, 2, 2, 2, 2, 6, 2, 3, 3])